# LeNet-5 Neural Network for MNIST

### Options

In [1]:
batch_size = 20
training_epochs = 5

### Load & convert data

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

def convert_data(x_data: np.ndarray, y_data: np.ndarray):
    # hacky way to increase the size of a (unknown)x28x28 to (unknown)x32x32
    # by appending two zeros at all sides of the 28x28 matrix
    tmp = np.zeros((x_data.shape[0], 2, x_data.shape[2]))
    x_data = np.append(x_data, tmp, axis=1)
    x_data = np.append(tmp, x_data, axis=1)
    tmp = np.zeros((x_data.shape[0], x_data.shape[1], 2))
    x_data = np.append(x_data, tmp, axis=2)
    x_data = np.append(tmp, x_data, axis=2)

    # change dimensions: (unknown)x32x32 to (unknown)x32x32x1
    x_data = np.expand_dims(x_data, -1)
    # change dimensions: (unknown) to (unknown)x10
    y_data = keras.utils.to_categorical(y_data, 10)

    return (x_data - 128, y_data)


# for now we are just interested in the training data and not the test data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

(X_train, y_train) = convert_data(X_train, y_train)
(X_test, y_test) = convert_data(X_test, y_test)


2024-11-14 22:55:57.286862: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731621357.372735   22097 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731621357.401516   22097 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-14 22:55:57.614890: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# save data
np.save("export/mnist/x_train.npy", X_train)
np.save("export/mnist/y_train.npy", y_train)
np.save("export/mnist/x_test.npy", X_test)
np.save("export/mnist/y_test.npy", y_test)

In [17]:
print("Shapes:")
print((X_train.shape, y_train.shape))

Shapes:
((60000, 32, 32, 1), (60000, 10))


### Setup our model

In [5]:
model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 1)),
    keras.layers.Conv2D(6, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(16, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(120),
    keras.layers.Dense(84),
    keras.layers.Dense(10),
    keras.layers.Activation('softmax')
])

model.summary()

2024-11-14 22:56:04.496421: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

### Fitting the model

In [6]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

hist = model.fit(X_train, y_train, batch_size=batch_size, epochs=training_epochs, validation_split=0.2)

Epoch 1/5


2024-11-14 22:56:05.356099: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 196608000 exceeds 10% of free system memory.


2400/2400 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.8414 - loss: 3.0447 - val_accuracy: 0.9485 - val_loss: 0.2156
Epoch 2/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9498 - loss: 0.1914 - val_accuracy: 0.9613 - val_loss: 0.1537
Epoch 3/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9575 - loss: 0.1666 - val_accuracy: 0.9577 - val_loss: 0.1704
Epoch 4/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9630 - loss: 0.1325 - val_accuracy: 0.9729 - val_loss: 0.0910
Epoch 5/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9690 - loss: 0.1056 - val_accuracy: 0.9712 - val_loss: 0.1053


## How good is our model for non-train data?

In [7]:
score = model.evaluate(X_test, y_test, batch_size=batch_size)

score

500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9629 - loss: 0.1280


[0.1019512414932251, 0.9700999855995178]

In [8]:
model.predict(X_train[3:5])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step


array([[1.1516418e-09, 9.9998260e-01, 1.4831263e-06, 1.7672818e-11,
        5.3299982e-06, 2.7092235e-11, 1.9051745e-10, 6.0989112e-07,
        9.9676381e-06, 3.8712651e-09],
       [3.5984219e-12, 3.9999247e-11, 7.5752782e-15, 1.5344134e-10,
        8.3611945e-07, 3.1900660e-10, 5.7835816e-15, 1.3022385e-08,
        6.9741535e-10, 9.9999917e-01]], dtype=float32)

In [9]:
from pathlib import Path

Path("export/").mkdir(exist_ok=True)
model.save("export/original_model.keras")

In [10]:
# import tensorflow_model_optimization as tfmot

# quantize_model = tfmot.quantization.keras.quantize_model
# q_aware_model = quantize_model(model)

In [11]:
# q_aware_model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

# q_aware_model.summary()

In [12]:
# hist = model.fit(X_train, y_train, batch_size=batch_size, epochs=training_epochs, validation_split=0.2)

In [13]:
# converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]

# tflite_qaware_model = converter.convert()


In [14]:
# with open("export/quantization_aware_training.tflite", 'wb') as f:
#     f.write(tflite_qaware_model)